In [1]:
!uv pip install -q \
    "scikit-learn" \
    numpy \
    pandas \
    pyaml \
    "sagemaker>=2.0.0,<3.0.0" \
    "sagemaker[local]" \
    moto

In [2]:
!docker build -q -t sagemaker-scikit-learn-processing-local container/.

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

sha256:250603795ae564d3178dca1cad7fed7da57b4d903345bc4daa4fbab36b312252


In [3]:
from sagemaker.processing import (
    ScriptProcessor,
    ProcessingInput,
    ProcessingOutput
)
from sagemaker_utils import get_local_session


local_session = get_local_session()


role = 'arn:aws:iam::111111111111:role/service-role/AmazonSageMaker-ExecutionRole-20200101T000001'

processor = ScriptProcessor(
    command=['python3'],
    image_uri='sagemaker-scikit-learn-processing-local',
    role=role,
    instance_count=1,
    instance_type='local',
    sagemaker_session=local_session
)

processor.run(
    code='processing_script.py',
    inputs=[ProcessingInput(
        source='./input_data/',
        destination='/opt/ml/processing/input_data/')],
    outputs=[ProcessingOutput(
        output_name='word_count_data',
        source='/opt/ml/processing/processed_data/')],
    arguments=['job-type', 'word-count']
)

preprocessing_job_description = processor.jobs[-1].describe()
output_config = preprocessing_job_description['ProcessingOutputConfig']

print(output_config)

for output in output_config['Outputs']:
    if output['OutputName'] == 'word_count_data':
        word_count_data_file = output['S3Output']['S3Uri']

print('Output file is located on: {}'.format(word_count_data_file))

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-processing-local-2026-03-13-20-18-01-545
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    name: sagemaker-local
services:
  algo-1-51v2b:
    container_name: y2shtge3gp-algo-1-51v2b
    entrypoint:
    - python3
    - /opt/ml/processing/input/code/pro

time="2026-03-13T20:18:02Z" level=warning msg="/tmp/tmpahrkfum8/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-13T20:18:02Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpahrkfum8\".\nSet `external: true` to use an existing network"
 Container y2shtge3gp-algo-1-51v2b Creating 
 Container y2shtge3gp-algo-1-51v2b Created 
Attaching to y2shtge3gp-algo-1-51v2b
 Container y2shtge3gp-algo-1-51v2b Starting 
 Container y2shtge3gp-algo-1-51v2b Started 
y2shtge3gp-algo-1-51v2b  | Processing Started
y2shtge3gp-algo-1-51v2b  | Received arguments {'job-type': 'word-count'}
y2shtge3gp-algo-1-51v2b  | Reading input data from /opt/ml/processing/input_data/
y2shtge3gp-algo-1-51v2b  | Got Args: {'job-type': 'word-count'}
y2shtge3gp-algo-1-51v2b  | Available input text files: ['sample_file_1.txt', 'sample_file_2.txt', 'sample_file_3.txt']
y2shtge3gp-algo-1-51v2b

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


y2shtge3gp-algo-1-51v2b exited with code 0
 Compose Stopping Aborting on container exit...
 Container y2shtge3gp-algo-1-51v2b Stopping 
 Container y2shtge3gp-algo-1-51v2b Stopped 
{'Outputs': [{'OutputName': 'word_count_data', 'AppManaged': False, 'S3Output': {'S3Uri': 's3://local-mock-bucket/sagemaker-scikit-learn-processing-local-2026-03-13-20-18-01-545/output/word_count_data', 'LocalPath': '/opt/ml/processing/processed_data/', 'S3UploadMode': 'EndOfJob'}}]}
Output file is located on: s3://local-mock-bucket/sagemaker-scikit-learn-processing-local-2026-03-13-20-18-01-545/output/word_count_data


In [4]:
# List objects in the mock bucket to confirm upload
response = local_session._s3_client.list_objects_v2(Bucket="local-mock-bucket")

print("Files in LocalStack S3:")
for obj in response.get('Contents', []):
    print(f" - {obj['Key']}")

# Optional: Download and read the result
output_key = [obj['Key'] for obj in response.get('Contents', []) if 'total_words' in obj['Key']][0]
local_session._s3_client.download_file("local-mock-bucket", output_key, "local_result.txt")

with open("local_result.txt", "r") as f:
    print(f"\nContent of result file:\n{f.read()}")

Files in LocalStack S3:
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-00-29-444/input/code/processing_script.py
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-00-52-255/input/code/processing_script.py
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/code/processing_script.py
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/.ipynb_checkpoints/sample_file_1-checkpoint.txt
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/.ipynb_checkpoints/sample_file_2-checkpoint.txt
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/.ipynb_checkpoints/sample_file_3-checkpoint.txt
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/sample_file_1.txt
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/sample_file_2.txt
 - sagemaker-scikit-learn-processing-local-2026-03-13-18-01-12-921/input/input-1/sample_file_3